# Random Forest — Predicción de Anemia Infantil (ENDES RECH6 2024)

**Variable objetivo:** `Anemia` (Sí / No), construida a partir de `HC57A`
(Nivel de Anemia, nueva directriz OMS 2024 / RM 251-2024-MINSA):
- `HC57A` en {1 Grave, 2 Moderada, 3 Leve} → **Anemia = Sí**
- `HC57A` = 4 (Sin anemia) → **Anemia = No**
- `HC57A` = 9 corresponde a niños sin hemoglobina medida → se excluyen.



In [1]:
# ============================================================
# RANDOM FOREST - PREDICCIÓN DE ANEMIA INFANTIL (ENDES RECH6 2024)
# ============================================================
# Adaptado del ejemplo de enfermedades cardíacas. Mejoras incluidas:
#   - class_weight='balanced' / 'balanced_subsample' para desbalance
#   - Out-of-Bag (OOB) score como validación interna
#   - GridSearchCV ampliado (criterion, class_weight, bootstrap)
#   - Comparativa con ExtraTreesClassifier (variante de RF)
#   - Importancia de variables por permutación (más robusta)
#   - Optimización de umbral (Youden's J y F1)
#   - Calibración de probabilidades (isotónica / sigmoide)
#   - Validación cruzada multi-métrica
#   - Learning curves
#   - Top-N features (subset model)
#   - Curvas Precision-Recall
#   - Predicción con encoders persistidos (sin librerías de serialización)
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split, cross_val_score, cross_validate,
    GridSearchCV, StratifiedKFold, learning_curve
)
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier
)
from sklearn.inspection import permutation_importance
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, accuracy_score,
    precision_recall_curve, average_precision_score,
    f1_score, precision_score, recall_score, brier_score_loss
)
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings("ignore")

# Reproducibilidad
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 0. Carga de datos

`RECH6_2024.xlsx` es el cuestionario individual de niños de ENDES 2024
(diccionario `Diccionario_-_RECH6.pdf`). Cada fila es un niño menor de 5 años
con sus mediciones antropométricas y de hemoglobina.



In [2]:
# --- Opción A: subir el archivo manualmente en Colab ---
try:
    from google.colab import files
    subido = files.upload()  # selecciona RECH6_2024.xlsx
    RUTA_ARCHIVO = list(subido.keys())[0]
except ImportError:
    # Estamos fuera de Colab (ej. Jupyter local): usar archivo en el mismo directorio
    RUTA_ARCHIVO = "RECH6_2024.xlsx"

# --- Opción B: montar Google Drive (alternativa, descomentar si prefieres) ---
# from google.colab import drive
# drive.mount('/content/drive')
# RUTA_ARCHIVO = "/content/drive/MyDrive/ruta/a/RECH6_2024.xlsx"

df_raw = pd.read_excel(RUTA_ARCHIVO)

print("=" * 60)
print("FORMA DEL DATASET (crudo):", df_raw.shape)
print("=" * 60)
print("\nPrimeras filas:")
print(df_raw.head())
print("\nDistribución de HC57A (nivel de anemia, incluye 9 = no medido):")
print(df_raw["HC57A"].value_counts())


Saving RECH6_2024.xlsx to RECH6_2024.xlsx
FORMA DEL DATASET (crudo): (20290, 43)

Primeras filas:
    ID1       HHID  HC0  HC1  HC2  HC3   HC4  HC5   HC6   HC7  ...  HC60  \
0  2024  325503101    3   43  150  962  2619  -64  9747  4378  ...     2   
1  2024  325504701    4   13   91  714   512 -163  9379  2245  ...     2   
2  2024  325505001    3   19  119  802  2107  -80  9695  7357  ...     1   
3  2024  325508901    4   13  100  747  3636  -35  9867  5619  ...     2   
4  2024  325509701    4   18  112  795  1358 -110  9593  3804  ...     2   

   HC61  HC62  HC63  HC64  HC68 HC70  HC71  HC72  HC73  
0     2     5    60     3     2  -88   -10    60    64  
1     2     5   130     2     2 -167   -17    81   111  
2     1     6   216     2     1  -78    91   175   190  
3     2     5   147     2     2  -27    67   104   112  
4     2     3    96     2     2 -119    13    93   119  

[5 rows x 43 columns]

Distribución de HC57A (nivel de anemia, incluye 9 = no medido):
HC57A
4    1259

## 1. Limpieza y construcción de variables

El cuestionario ENDES codifica los valores faltantes/no aplicables con
códigos especiales (`9999`, `9998`, `999`, `' '` en blanco, etc.) y varias
variables numéricas vienen multiplicadas por 10 o por 100 (para guardarlas
como enteros con decimales). Aquí:

1. Nos quedamos solo con niños a los que **sí se les midió hemoglobina**
   (`HC55 == 0`), que es exactamente el subconjunto con `HC57A` en {1,2,3,4}.
2. Construimos la variable objetivo binaria `Anemia` (Sí/No).
3. Convertimos peso, talla y los puntajes Z (talla/edad, peso/edad,
   peso/talla, IMC/edad) a sus unidades reales, tratando los códigos de
   "no plausible" como nulos.
4. Construimos variables de la madre (nivel educativo, años de educación,
   intervalo desde el nacimiento anterior) manejando los códigos especiales
   (`8` = No sabe, `999` = no aplica, espacios en blanco = primer hijo).
5. **Excluimos explícitamente** `HC53`, `HC55`, `HC56`, `HC56A`, `HC57`,
   `HC57A` como *features*: son la propia hemoglobina o el nivel de anemia
   ya calculado, así que usarlas sería una fuga de información (*data
   leakage*) — el modelo "adivinaría" la respuesta en vez de aprender a
   predecirla a partir de variables demográficas/antropométricas.


In [3]:
df = df_raw.copy()

# 1) Solo niños con hemoglobina efectivamente medida
df = df[df["HC55"] == 0].copy()

# 2) Variable objetivo binaria
#    HC57A: 1 Grave, 2 Moderada, 3 Leve, 4 Sin anemia
df["Anemia"] = np.where(df["HC57A"].isin([1, 2, 3]), "Anemia", "Sin anemia")

# 3) Antropometría a unidades reales
df["Peso_kg"]  = df["HC2"].replace(9999, np.nan) / 10          # kg, 1 decimal
df["Talla_cm"] = df["HC3"].replace(9999, np.nan) / 10          # cm, 1 decimal

for col in ["HC70", "HC71", "HC72", "HC73"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").replace(9999, np.nan) / 100

df = df.rename(columns={
    "HC70": "ZScore_Talla_Edad",
    "HC71": "ZScore_Peso_Edad",
    "HC72": "ZScore_Peso_Talla",
    "HC73": "ZScore_IMC_Edad",
})

# 4) Variables maternas / de nacimiento
df["Educacion_Madre_Anios"] = pd.to_numeric(df["HC62"].replace(" ", np.nan), errors="coerce")
df["Educacion_Madre_Anios"] = df["Educacion_Madre_Anios"].fillna(df["Educacion_Madre_Anios"].median())

intervalo = pd.to_numeric(df["HC63"].replace(" ", np.nan), errors="coerce").replace(999, np.nan)
df["Es_Primerizo"] = intervalo.isnull().astype(int)          # 1 = no hay nacimiento previo
df["Intervalo_Nacimiento_Meses"] = intervalo.fillna(0)        # 0 cuando es primerizo

df["Orden_Nacimiento"] = pd.to_numeric(df["HC64"], errors="coerce")

nivel_educ = pd.to_numeric(df["HC61"], errors="coerce").replace(8, np.nan)  # 8 = No sabe
mapa_nivel = {0: "Sin educacion", 1: "Primaria", 2: "Secundaria", 3: "Superior"}
df["Educacion_Madre_Nivel"] = nivel_educ.map(mapa_nivel)
df["Educacion_Madre_Nivel"] = df["Educacion_Madre_Nivel"].fillna("Sin educacion")

df["Sexo"] = df["HC27"].map({1: "Hombre", 2: "Mujer"})
df["Edad_meses"] = df["HC1"]
df["Mes_Nacimiento"] = df["HC30"]

# Columnas finales del modelo (excluye HC53/HC55/HC56/HC56A/HC57/HC57A: fuga de información)
columnas_modelo = [
    "Edad_meses", "Sexo", "Peso_kg", "Talla_cm",
    "ZScore_Talla_Edad", "ZScore_Peso_Edad", "ZScore_Peso_Talla", "ZScore_IMC_Edad",
    "Educacion_Madre_Nivel", "Educacion_Madre_Anios",
    "Intervalo_Nacimiento_Meses", "Es_Primerizo", "Orden_Nacimiento", "Mes_Nacimiento",
    "Anemia",
]

df_model = df[columnas_modelo].dropna().reset_index(drop=True)

print("=" * 60)
print("FORMA DEL DATASET (listo para modelar):", df_model.shape)
print("=" * 60)
print("\nPrimeras filas:")
print(df_model.head())
print("\nTipos de datos:")
print(df_model.dtypes)
print("\nValores nulos por columna:")
nulos = df_model.isnull().sum()
print(nulos[nulos > 0] if nulos.sum() > 0 else "  Ninguno")
print("\nDistribución de la variable objetivo:")
print(df_model["Anemia"].value_counts())
print(df_model["Anemia"].value_counts(normalize=True).mul(100).round(2).astype(str) + "%")
print("\nCardinalidad de columnas categóricas:")
for col in df_model.select_dtypes(include=["object"]).columns:
    print(f"  {col}: {df_model[col].nunique()} categorías -> {sorted(df_model[col].unique())}")


FORMA DEL DATASET (listo para modelar): (18342, 15)

Primeras filas:
   Edad_meses    Sexo  Peso_kg  Talla_cm  ZScore_Talla_Edad  ZScore_Peso_Edad  \
0          43   Mujer     15.0      96.2              -0.88             -0.10   
1          13   Mujer      9.1      71.4              -1.67             -0.17   
2          19   Mujer     11.9      80.2              -0.78              0.91   
3          13   Mujer     10.0      74.7              -0.27              0.67   
4          18  Hombre     11.2      79.5              -1.19              0.13   

   ZScore_Peso_Talla  ZScore_IMC_Edad Educacion_Madre_Nivel  \
0               0.60             0.64            Secundaria   
1               0.81             1.11            Secundaria   
2               1.75             1.90              Primaria   
3               1.04             1.12            Secundaria   
4               0.93             1.19            Secundaria   

   Educacion_Madre_Anios  Intervalo_Nacimiento_Meses  Es_Primeriz

## 2. Preprocesamiento

**Mejora clave:** guardamos los `LabelEncoder` ajustados en un diccionario
para reutilizarlos en predicciones nuevas (evita que un encoder distinto en
la celda de predicción genere un *mapping* incompatible con el de
entrenamiento).


In [4]:
df = df_model  # alias para mantener el resto del notebook igual al original

# Separar features y target
X = df.drop(columns=["Anemia"])
y = df["Anemia"]

# División train/test (80/20, estratificada)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Train: {X_train.shape[0]} muestras | Test: {X_test.shape[0]} muestras")
print(f"Distribución train: {y_train.value_counts(normalize=True).round(3).to_dict()}")
print(f"Distribución test : {y_test.value_counts(normalize=True).round(3).to_dict()}")

# Codificación categórica con encoders persistidos
label_encoders = {}
categorical_cols = X_train.select_dtypes(include=["object"]).columns.tolist()

for col in categorical_cols:
    le = LabelEncoder()
    # Ajustamos sobre TODO el dataset (no solo train) para conocer todas las
    # categorías posibles — así no fallamos con valores nuevos no vistos en train.
    le.fit(df[col].astype(str))
    X_train[col] = le.transform(X_train[col].astype(str))
    X_test[col]  = le.transform(X_test[col].astype(str))
    label_encoders[col] = le

# Codificar variable objetivo
# OJO: LabelEncoder ordena alfabéticamente ("Anemia" < "Sin anemia"), lo que
# dejaría "Anemia" = 0 y volvería ambigua la lectura de predict_proba(...)[:, 1]
# en el resto del notebook. Para que la clase positiva de interés clínico
# ("Anemia") sea siempre el índice 1, usamos un encoder simple con orden fijo.
class TargetEncoder:
    """Encoder de 2 clases con orden fijo (no alfabético) y misma API que LabelEncoder."""
    def __init__(self, clases_ordenadas):
        self.classes_ = np.array(clases_ordenadas)
        self._map = {c: i for i, c in enumerate(self.classes_)}
    def transform(self, valores):
        return np.array([self._map[v] for v in valores])
    def inverse_transform(self, indices):
        return self.classes_[np.asarray(indices)]

le_y = TargetEncoder(["Sin anemia", "Anemia"])  # índice 0 = Sin anemia, índice 1 = Anemia
y_train_encoded = le_y.transform(y_train.astype(str))
y_test_encoded  = le_y.transform(y_test.astype(str))

print(f"\nColumnas categóricas codificadas: {categorical_cols}")
print(f"Mapeo del target: {dict(zip(le_y.classes_, le_y.transform(le_y.classes_)))}")
print(f"\n✓ Encoders persistidos en `label_encoders` para uso futuro.")


Train: 14673 muestras | Test: 3669 muestras
Distribución train: {'Sin anemia': 0.686, 'Anemia': 0.314}
Distribución test : {'Sin anemia': 0.686, 'Anemia': 0.314}

Columnas categóricas codificadas: ['Sexo', 'Educacion_Madre_Nivel']
Mapeo del target: {np.str_('Sin anemia'): np.int64(0), np.str_('Anemia'): np.int64(1)}

✓ Encoders persistidos en `label_encoders` para uso futuro.


## 3. Modelo Base (Random Forest)

Modelo de referencia con configuración por defecto para tener una línea base
contra la que comparar las mejoras.


In [5]:
rf_base = RandomForestClassifier(
    n_estimators=100,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_base.fit(X_train, y_train_encoded)

y_pred_base = rf_base.predict(X_test)
y_prob_base = rf_base.predict_proba(X_test)[:, 1]

print("=" * 60)
print("MODELO BASE - RESULTADOS")
print("=" * 60)
print(f"Accuracy : {accuracy_score(y_test_encoded, y_pred_base):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test_encoded, y_prob_base):.4f}")
print(f"F1-Score : {f1_score(y_test_encoded, y_pred_base):.4f}")
print(f"\nReporte de clasificación:")
print(classification_report(y_test_encoded, y_pred_base, target_names=le_y.classes_))


MODELO BASE - RESULTADOS
Accuracy : 0.6822
ROC-AUC  : 0.6159
F1-Score : 0.2667

Reporte de clasificación:
              precision    recall  f1-score   support

  Sin anemia       0.71      0.91      0.80      2517
      Anemia       0.48      0.18      0.27      1152

    accuracy                           0.68      3669
   macro avg       0.60      0.55      0.53      3669
weighted avg       0.64      0.68      0.63      3669



## 4. Out-of-Bag (OOB) Score

Random Forest permite evaluación interna **sin necesidad de un conjunto de
validación separado**: cada árbol se evalúa sobre las muestras que **no**
vio durante el bootstrap (~37%). Activamos también `class_weight='balanced'`
para mitigar el desbalance de clases (~69% sin anemia / 31% con anemia).


In [6]:
# RF con OOB y class_weight balanceado
rf_oob = RandomForestClassifier(
    n_estimators=300,
    max_features="sqrt",
    oob_score=True,
    bootstrap=True,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_oob.fit(X_train, y_train_encoded)

print("=" * 60)
print("RANDOM FOREST + OOB + class_weight='balanced'")
print("=" * 60)
print(f"OOB Score (accuracy) : {rf_oob.oob_score_:.4f}")

# Validamos también en test
y_pred_oob = rf_oob.predict(X_test)
y_prob_oob = rf_oob.predict_proba(X_test)[:, 1]
print(f"Test Accuracy        : {accuracy_score(y_test_encoded, y_pred_oob):.4f}")
print(f"Test ROC-AUC         : {roc_auc_score(y_test_encoded, y_prob_oob):.4f}")
print(f"Test F1              : {f1_score(y_test_encoded, y_pred_oob):.4f}")


RANDOM FOREST + OOB + class_weight='balanced'
OOB Score (accuracy) : 0.6837
Test Accuracy        : 0.6868
Test ROC-AUC         : 0.6181
Test F1              : 0.2475


## 5. Optimización con GridSearchCV

Buscamos la mejor combinación de hiperparámetros:
- `class_weight` (manejo de desbalance)
- `criterion` (función de impureza: gini / log_loss)
- `min_samples_split` y `min_samples_leaf`
- `n_estimators` (200 / 400)

El grid está dimensionado para ejecutarse en unos pocos minutos en Colab.


In [ ]:
param_grid = {
    "n_estimators":      [200, 400],
    "max_depth":         [None, 10, 15],
    "min_samples_split": [2, 5],
    "min_samples_leaf":  [1, 2],
    "max_features":      ["sqrt", "log2"],
    "criterion":         ["gini", "log_loss"],
    "class_weight":      [None, "balanced"],
    "bootstrap":         [True],
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring="roc_auc",
    n_jobs=-1,
    verbose=1,
    refit=True,
)
grid_search.fit(X_train, y_train_encoded)

print("\n" + "=" * 60)
print("MEJORES HIPERPARÁMETROS")
print("=" * 60)
print(grid_search.best_params_)
print(f"Mejor ROC-AUC (CV): {grid_search.best_score_:.4f}")

# Guardamos el mejor modelo
rf_best = grid_search.best_estimator_


Fitting 5 folds for each of 192 candidates, totalling 960 fits


## 6. Modelo Optimizado — Evaluación en Test

In [ ]:
y_pred_best  = rf_best.predict(X_test)
y_prob_best  = rf_best.predict_proba(X_test)[:, 1]

print("=" * 60)
print("MODELO OPTIMIZADO - RESULTADOS EN TEST")
print("=" * 60)
print(f"Accuracy : {accuracy_score(y_test_encoded, y_pred_best):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test_encoded, y_prob_best):.4f}")
print(f"F1       : {f1_score(y_test_encoded, y_pred_best):.4f}")
print(f"Precision: {precision_score(y_test_encoded, y_pred_best):.4f}")
print(f"Recall   : {recall_score(y_test_encoded, y_pred_best):.4f}")
print("\nReporte de clasificación:")
print(classification_report(y_test_encoded, y_pred_best, target_names=le_y.classes_))


## 7. Validación Cruzada Multi-Métrica (5-Fold Estratificado)

Evaluamos el modelo optimizado con **múltiples métricas** simultáneamente
para obtener una visión más robusta y detectar síntomas de overfitting.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "accuracy":  "accuracy",
    "precision": "precision",
    "recall":    "recall",
    "f1":        "f1",
    "roc_auc":   "roc_auc",
}

cv_results = cross_validate(
    rf_best, X_train, y_train_encoded,
    cv=cv, scoring=scoring, n_jobs=-1, return_train_score=True
)

print("=" * 60)
print("VALIDACIÓN CRUZADA (5-Fold) — MODELO OPTIMIZADO")
print("=" * 60)
print(f"{'Métrica':10s}  {'Test (mean ± std)':25s}   {'Train (mean ± std)':25s}")
print("-" * 70)
for metric in scoring.keys():
    test_key  = f"test_{metric}"
    train_key = f"train_{metric}"
    test_str  = f"{cv_results[test_key].mean():.4f} ± {cv_results[test_key].std():.4f}"
    train_str = f"{cv_results[train_key].mean():.4f} ± {cv_results[train_key].std():.4f}"
    print(f"{metric.upper():10s}  {test_str:25s}   {train_str:25s}")


## 8. Comparativa con ExtraTreesClassifier

**Extra Trees** (Extremely Randomized Trees) es una variante de RF que
introduce aleatoriedad adicional en los puntos de corte. Comparamos ambos
bajo el mismo protocolo.


In [ ]:
et_model = ExtraTreesClassifier(
    n_estimators=300,
    max_depth=rf_best.max_depth,
    min_samples_split=rf_best.min_samples_split,
    min_samples_leaf=rf_best.min_samples_leaf,
    max_features=rf_best.max_features,
    class_weight=rf_best.class_weight,
    criterion=rf_best.criterion,
    bootstrap=False,  # ExtraTrees usa bootstrap=False por defecto
    random_state=RANDOM_STATE,
    n_jobs=-1
)
et_model.fit(X_train, y_train_encoded)

y_pred_et = et_model.predict(X_test)
y_prob_et = et_model.predict_proba(X_test)[:, 1]

et_cv = cross_validate(
    et_model, X_train, y_train_encoded,
    cv=cv, scoring=["accuracy", "roc_auc", "f1"], n_jobs=-1
)

print("=" * 60)
print("COMPARATIVA EN TEST  —  RF vs ExtraTrees")
print("=" * 60)
comp = pd.DataFrame({
    "Métrica":    ["Accuracy", "ROC-AUC", "F1", "Precision", "Recall"],
    "RandomForest": [
        accuracy_score(y_test_encoded, y_pred_best),
        roc_auc_score(y_test_encoded, y_prob_best),
        f1_score(y_test_encoded, y_pred_best),
        precision_score(y_test_encoded, y_pred_best),
        recall_score(y_test_encoded, y_pred_best),
    ],
    "ExtraTrees": [
        accuracy_score(y_test_encoded, y_pred_et),
        roc_auc_score(y_test_encoded, y_prob_et),
        f1_score(y_test_encoded, y_pred_et),
        precision_score(y_test_encoded, y_pred_et),
        recall_score(y_test_encoded, y_pred_et),
    ],
})
print(comp.to_string(index=False))

print("\nCV-ROC-AUC:")
print(f"  RandomForest  : {cv_results['test_roc_auc'].mean():.4f} ± {cv_results['test_roc_auc'].std():.4f}")
print(f"  ExtraTrees    : {et_cv['test_roc_auc'].mean():.4f} ± {et_cv['test_roc_auc'].std():.4f}")


## 9. Importancia por Permutación

La `feature_importances_` de sklearn mide la **disminución media de impureza
(Gini)**, que favorece a variables de alta cardinalidad o numéricas. La
**importancia por permutación** mide la caída real de performance al
desordenar cada variable → más robusta y menos sesgada.


In [ ]:
perm_imp = permutation_importance(
    rf_best, X_test, y_test_encoded,
    n_repeats=30,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    scoring="roc_auc"
)

perm_df = pd.DataFrame({
    "Feature":       X.columns,
    "Gini_Imp":      rf_best.feature_importances_,
    "Perm_Mean":     perm_imp.importances_mean,
    "Perm_Std":      perm_imp.importances_std,
}).sort_values("Perm_Mean", ascending=False)

print("=" * 60)
print("IMPORTANCIA DE VARIABLES (Gini vs Permutación)")
print("=" * 60)
print(perm_df.to_string(index=False))


## 10. Optimización del Umbral de Decisión

Por defecto se usa 0.5, pero el umbral óptimo depende del costo de falsos
positivos vs. falsos negativos (en salud, no detectar un niño con anemia
suele ser más costoso que una falsa alarma). Optimizamos con **F1 máximo** y
**Youden's J** (`TPR − FPR`).


In [ ]:
# Probabilidades del modelo optimizado
precisions, recalls, thresholds = precision_recall_curve(y_test_encoded, y_prob_best)
f1_scores_curve = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)

# F1 máximo (nota: thresholds tiene un elemento menos que precisions/recalls)
ix_f1 = np.argmax(f1_scores_curve[:-1])  # alineamos con thresholds
thr_f1 = thresholds[ix_f1]

# Youden's J = TPR - FPR
fpr, tpr, roc_thresholds = roc_curve(y_test_encoded, y_prob_best)
j_scores = tpr - fpr
ix_j = np.argmax(j_scores)
thr_j = roc_thresholds[ix_j]

print("=" * 60)
print("OPTIMIZACIÓN DE UMBRAL")
print("=" * 60)
print(f"Umbral por defecto (0.5):")
print(f"   F1 = {f1_score(y_test_encoded, y_pred_best):.4f}  |  "
      f"Recall = {recall_score(y_test_encoded, y_pred_best):.4f}  |  "
      f"Precision = {precision_score(y_test_encoded, y_pred_best):.4f}")
print(f"\nUmbral óptimo F1 ({thr_f1:.3f}):")
print(f"   F1 = {f1_scores_curve[ix_f1]:.4f}  |  "
      f"Recall = {recalls[ix_f1]:.4f}  |  "
      f"Precision = {precisions[ix_f1]:.4f}")
print(f"\nUmbral Youden's J ({thr_j:.3f}):")
print(f"   TPR = {tpr[ix_j]:.4f}  |  FPR = {fpr[ix_j]:.4f}  |  J = {j_scores[ix_j]:.4f}")

# Aplicamos el mejor umbral
y_pred_thr_f1 = (y_prob_best >= thr_f1).astype(int)
print("\nMétricas con umbral F1-óptimo:")
print(classification_report(y_test_encoded, y_pred_thr_f1, target_names=le_y.classes_))


## 11. Calibración de Probabilidades

Las probabilidades de los árboles (votación) no son probabilidades reales —
son **votos normalizados**. `CalibratedClassifierCV` ajusta un regresor
(isotónico o sigmoide/Platt) para obtener probabilidades calibradas — útil
si luego quieres reportar "% de riesgo de anemia" de forma confiable.


In [ ]:
# Calibración isotónica (no paramétrica) y sigmoide (Platt)
cal_iso = CalibratedClassifierCV(rf_best, method="isotonic", cv=5, n_jobs=-1)
cal_sig = CalibratedClassifierCV(rf_best, method="sigmoid", cv=5, n_jobs=-1)

cal_iso.fit(X_train, y_train_encoded)
cal_sig.fit(X_train, y_train_encoded)

y_prob_iso = cal_iso.predict_proba(X_test)[:, 1]
y_prob_sig = cal_sig.predict_proba(X_test)[:, 1]

brier_raw = brier_score_loss(y_test_encoded, y_prob_best)
brier_iso = brier_score_loss(y_test_encoded, y_prob_iso)
brier_sig = brier_score_loss(y_test_encoded, y_prob_sig)

print("=" * 60)
print("CALIBRACIÓN DE PROBABILIDADES")
print("=" * 60)
print(f"{'Versión':25s}  {'Brier ↓':10s}  {'ROC-AUC ↑':10s}")
print("-" * 50)
for nombre, brier, yp in [("RF crudo",  brier_raw, y_prob_best),
                          ("Isotónica", brier_iso, y_prob_iso),
                          ("Sigmoide",  brier_sig, y_prob_sig)]:
    auc = roc_auc_score(y_test_encoded, yp)
    print(f"{nombre:25s}  {brier:10.4f}  {auc:10.4f}")


## 12. Modelo con Top-N Features

Entrenamos un RF únicamente con las top-N variables más importantes (por
permutación) y comparamos su rendimiento. Útil para reducir varianza y
mejorar la interpretabilidad (por ejemplo, para una ficha clínica
simplificada).


In [ ]:
top_features = perm_df.sort_values("Perm_Mean", ascending=False)["Feature"].tolist()

print("=" * 60)
print("RENDIMIENTO POR NÚMERO DE FEATURES (Top-N)")
print("=" * 60)
results_topn = []
for k in range(3, len(top_features) + 1):
    feats = top_features[:k]
    rf_sub = RandomForestClassifier(
        n_estimators=300,
        max_depth=rf_best.max_depth,
        min_samples_split=rf_best.min_samples_split,
        min_samples_leaf=rf_best.min_samples_leaf,
        class_weight=rf_best.class_weight,
        criterion=rf_best.criterion,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    cv_sub = cross_val_score(rf_sub, X_train[feats], y_train_encoded,
                             cv=cv, scoring="roc_auc", n_jobs=-1)
    results_topn.append({"k": k, "roc_auc_mean": cv_sub.mean(), "roc_auc_std": cv_sub.std()})
    print(f"  k={k:2d}  ROC-AUC = {cv_sub.mean():.4f} ± {cv_sub.std():.4f}  | features = {feats}")

topn_df = pd.DataFrame(results_topn)
best_k = int(topn_df.loc[topn_df["roc_auc_mean"].idxmax(), "k"])
print(f"\n✓ Mejor k = {best_k} features")


## 13. Learning Curves

Para detectar **overfitting/underfitting** graficamos la evolución de train
y validación a medida que crece el tamaño del conjunto de entrenamiento.


In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    rf_best, X_train, y_train_encoded,
    cv=cv, scoring="roc_auc",
    train_sizes=np.linspace(0.1, 1.0, 10),
    n_jobs=-1, random_state=RANDOM_STATE
)

train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_mean   = val_scores.mean(axis=1)
val_std    = val_scores.std(axis=1)

print("✓ Learning curves calculadas. Se visualizarán en la sección de gráficos.")
print(f"  Gap train-val final: {train_mean[-1] - val_mean[-1]:.4f}")
if (train_mean[-1] - val_mean[-1]) > 0.05:
    print("  → Hay algo de overfitting: el modelo aprovecha señales del train que no generalizan.")
else:
    print("  → Gap train-val bajo: el modelo generaliza razonablemente bien.")


## 14. Visualizaciones

In [ ]:
fig = plt.figure(figsize=(20, 22))
gs = fig.add_gridspec(5, 3, hspace=0.45, wspace=0.3)
fig.suptitle("Random Forest Mejorado — Predicción de Anemia Infantil (ENDES RECH6 2024)",
             fontsize=18, fontweight="bold", y=0.995)

# 14.1 Matriz de Confusión — Base
ax1 = fig.add_subplot(gs[0, 0])
cm_base = confusion_matrix(y_test_encoded, y_pred_base)
sns.heatmap(cm_base, annot=True, fmt="d", cmap="Blues",
            xticklabels=le_y.classes_, yticklabels=le_y.classes_, ax=ax1, cbar=False)
ax1.set_title("Matriz de Confusión — Base", fontweight="bold")
ax1.set_ylabel("Real"); ax1.set_xlabel("Predicho")

# 14.2 Matriz de Confusión — Optimizado
ax2 = fig.add_subplot(gs[0, 1])
cm_best = confusion_matrix(y_test_encoded, y_pred_best)
sns.heatmap(cm_best, annot=True, fmt="d", cmap="Greens",
            xticklabels=le_y.classes_, yticklabels=le_y.classes_, ax=ax2, cbar=False)
ax2.set_title("Matriz de Confusión — Optimizado", fontweight="bold")
ax2.set_ylabel("Real"); ax2.set_xlabel("Predicho")

# 14.3 Matriz de Confusión — Umbral F1
ax3 = fig.add_subplot(gs[0, 2])
cm_thr = confusion_matrix(y_test_encoded, y_pred_thr_f1)
sns.heatmap(cm_thr, annot=True, fmt="d", cmap="Oranges",
            xticklabels=le_y.classes_, yticklabels=le_y.classes_, ax=ax3, cbar=False)
ax3.set_title(f"Matriz de Confusión — Umbral F1={thr_f1:.2f}", fontweight="bold")
ax3.set_ylabel("Real"); ax3.set_xlabel("Predicho")

# 14.4 Curva ROC
ax4 = fig.add_subplot(gs[1, 0])
fpr_b, tpr_b, _ = roc_curve(y_test_encoded, y_prob_base)
fpr_o, tpr_o, _ = roc_curve(y_test_encoded, y_prob_oob)
fpr_x, tpr_x, _ = roc_curve(y_test_encoded, y_prob_best)
fpr_e, tpr_e, _ = roc_curve(y_test_encoded, y_prob_et)
ax4.plot(fpr_b, tpr_b, lw=1.5, label=f"Base (AUC={roc_auc_score(y_test_encoded, y_prob_base):.3f})")
ax4.plot(fpr_o, tpr_o, lw=1.5, label=f"OOB balanced (AUC={roc_auc_score(y_test_encoded, y_prob_oob):.3f})")
ax4.plot(fpr_x, tpr_x, lw=2.5, label=f"Optimizado (AUC={roc_auc_score(y_test_encoded, y_prob_best):.3f})", color="darkorange")
ax4.plot(fpr_e, tpr_e, lw=1.5, linestyle="--", label=f"ExtraTrees (AUC={roc_auc_score(y_test_encoded, y_prob_et):.3f})")
ax4.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5)
ax4.set_xlabel("FPR"); ax4.set_ylabel("TPR")
ax4.set_title("Curva ROC", fontweight="bold")
ax4.legend(loc="lower right", fontsize=8); ax4.grid(alpha=0.3)

# 14.5 Curva Precision-Recall
ax5 = fig.add_subplot(gs[1, 1])
ap_best = average_precision_score(y_test_encoded, y_prob_best)
ap_et   = average_precision_score(y_test_encoded, y_prob_et)
ax5.plot(recalls, precisions, lw=2, color="darkorange",
         label=f"Optimizado (AP={ap_best:.3f})")
prec_et, rec_et, _ = precision_recall_curve(y_test_encoded, y_prob_et)
ax5.plot(rec_et, prec_et, lw=1.5, linestyle="--",
         label=f"ExtraTrees (AP={ap_et:.3f})")
ax5.axvline(thr_f1, ls=":", color="red", label=f"Umbral F1={thr_f1:.2f}")
ax5.set_xlabel("Recall"); ax5.set_ylabel("Precision")
ax5.set_title("Curva Precision-Recall", fontweight="bold")
ax5.legend(loc="lower left", fontsize=8); ax5.grid(alpha=0.3)

# 14.6 Importancia de variables (Permutación)
ax6 = fig.add_subplot(gs[1, 2])
perm_plot = perm_df.sort_values("Perm_Mean", ascending=True)
y_pos = np.arange(len(perm_plot))
ax6.barh(y_pos, perm_plot["Perm_Mean"], xerr=perm_plot["Perm_Std"], color="steelblue")
ax6.set_yticks(y_pos); ax6.set_yticklabels(perm_plot["Feature"], fontsize=8)
ax6.set_xlabel("Importancia (ROC-AUC drop)")
ax6.set_title("Importancia por Permutación", fontweight="bold")
ax6.grid(alpha=0.3, axis="x")

# 14.7 Comparación Gini vs Permutación (normalizadas)
ax7 = fig.add_subplot(gs[2, 0])
comp_imp = perm_df.copy()
comp_imp["Gini_norm"] = comp_imp["Gini_Imp"] / comp_imp["Gini_Imp"].sum()
comp_imp["Perm_norm"] = comp_imp["Perm_Mean"] / comp_imp["Perm_Mean"].sum()
comp_imp = comp_imp.sort_values("Perm_norm", ascending=True)
y_pos2 = np.arange(len(comp_imp))
width = 0.4
ax7.barh(y_pos2 - width/2, comp_imp["Gini_norm"], height=width, label="Gini", color="lightcoral")
ax7.barh(y_pos2 + width/2, comp_imp["Perm_norm"], height=width, label="Permutación", color="steelblue")
ax7.set_yticks(y_pos2); ax7.set_yticklabels(comp_imp["Feature"], fontsize=7)
ax7.set_title("Gini vs Permutación (normalizado)", fontweight="bold")
ax7.legend(fontsize=8); ax7.grid(alpha=0.3, axis="x")

# 14.8 Top-N features vs ROC-AUC
ax8 = fig.add_subplot(gs[2, 1])
ax8.plot(topn_df["k"], topn_df["roc_auc_mean"], marker="o", color="darkorange")
ax8.fill_between(topn_df["k"],
                  topn_df["roc_auc_mean"] - topn_df["roc_auc_std"],
                  topn_df["roc_auc_mean"] + topn_df["roc_auc_std"],
                  alpha=0.2, color="darkorange")
ax8.axvline(best_k, ls="--", color="red", label=f"Mejor k={best_k}")
ax8.set_xlabel("Número de features (k)"); ax8.set_ylabel("ROC-AUC (CV)")
ax8.set_title("Rendimiento vs Nº de Features", fontweight="bold")
ax8.legend(fontsize=8); ax8.grid(alpha=0.3)

# 14.9 Calibración: Brier score comparativo
ax9 = fig.add_subplot(gs[2, 2])
nombres_cal = ["RF crudo", "Isotónica", "Sigmoide"]
briers = [brier_raw, brier_iso, brier_sig]
colores_cal = ["gray", "seagreen", "steelblue"]
ax9.bar(nombres_cal, briers, color=colores_cal)
ax9.set_ylabel("Brier Score (↓ mejor)")
ax9.set_title("Calibración de Probabilidades", fontweight="bold")
ax9.grid(alpha=0.3, axis="y")

# 14.10 Learning curve
ax10 = fig.add_subplot(gs[3, :2])
ax10.plot(train_sizes, train_mean, "o-", color="steelblue", label="Train")
ax10.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color="steelblue")
ax10.plot(train_sizes, val_mean, "o-", color="darkorange", label="Validación (CV)")
ax10.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.15, color="darkorange")
ax10.set_xlabel("Tamaño del conjunto de entrenamiento")
ax10.set_ylabel("ROC-AUC")
ax10.set_title("Learning Curves — Modelo Optimizado", fontweight="bold")
ax10.legend(fontsize=9); ax10.grid(alpha=0.3)

# 14.11 Distribución de Anemia por Z-Score Talla/Edad
ax11 = fig.add_subplot(gs[3, 2])
for etiqueta in df["Anemia"].unique():
    subset = df[df["Anemia"] == etiqueta]["ZScore_Talla_Edad"]
    sns.kdeplot(subset, ax=ax11, label=etiqueta, fill=True, alpha=0.3)
ax11.set_xlabel("Z-Score Talla/Edad")
ax11.set_title("Distribución Z-Talla/Edad según Anemia", fontweight="bold")
ax11.legend(fontsize=8); ax11.grid(alpha=0.3)

# 14.12 Distribución de probabilidad predicha (modelo optimizado)
ax12 = fig.add_subplot(gs[4, 0])
ax12.hist(y_prob_best[y_test_encoded == 0], bins=25, alpha=0.6, label="Sin anemia (real)", color="steelblue")
ax12.hist(y_prob_best[y_test_encoded == 1], bins=25, alpha=0.6, label="Anemia (real)", color="darkorange")
ax12.axvline(0.5, ls="--", color="gray", label="Umbral 0.5")
ax12.axvline(thr_f1, ls=":", color="red", label=f"Umbral F1={thr_f1:.2f}")
ax12.set_xlabel("Probabilidad predicha de Anemia")
ax12.set_ylabel("Frecuencia")
ax12.set_title("Distribución de Probabilidades Predichas", fontweight="bold")
ax12.legend(fontsize=7); ax12.grid(alpha=0.3)

# 14.13 Edad (meses) vs Anemia
ax13 = fig.add_subplot(gs[4, 1])
sns.boxplot(data=df, x="Anemia", y="Edad_meses", ax=ax13, palette=["steelblue", "darkorange"])
ax13.set_title("Edad (meses) según Anemia", fontweight="bold")
ax13.grid(alpha=0.3, axis="y")

# 14.14 Comparativa RF vs ExtraTrees (barras)
ax14 = fig.add_subplot(gs[4, 2])
metricas_barra = ["Accuracy", "ROC-AUC", "F1", "Precision", "Recall"]
rf_vals = comp["RandomForest"].values
et_vals = comp["ExtraTrees"].values
x_pos = np.arange(len(metricas_barra))
width = 0.35
ax14.bar(x_pos - width/2, rf_vals, width, label="RandomForest", color="darkorange")
ax14.bar(x_pos + width/2, et_vals, width, label="ExtraTrees", color="seagreen")
ax14.set_xticks(x_pos); ax14.set_xticklabels(metricas_barra, rotation=30, fontsize=8)
ax14.set_title("RF vs ExtraTrees", fontweight="bold")
ax14.legend(fontsize=8); ax14.grid(alpha=0.3, axis="y")

plt.show()


## 15. Resumen Final Comparativo

In [ ]:
print("\n" + "=" * 60)
print("RESUMEN COMPARATIVO — TODOS LOS MODELOS")
print("=" * 60)

def metricas_completas(y_true, y_pred, y_prob):
    return {
        "Accuracy":  accuracy_score(y_true, y_pred),
        "ROC-AUC":   roc_auc_score(y_true, y_prob),
        "F1":        f1_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall":    recall_score(y_true, y_pred),
    }

resumen = pd.DataFrame({
    "Base":                metricas_completas(y_test_encoded, y_pred_base, y_prob_base),
    "OOB balanced":        metricas_completas(y_test_encoded, y_pred_oob,  y_prob_oob),
    "Optimizado (Grid)":   metricas_completas(y_test_encoded, y_pred_best, y_prob_best),
    "ExtraTrees":          metricas_completas(y_test_encoded, y_pred_et,   y_prob_et),
    f"Umbral F1={thr_f1:.2f}": metricas_completas(y_test_encoded, y_pred_thr_f1, y_prob_best),
}).T.round(4)

cv_summary = pd.DataFrame({
    "Base":                [np.nan, np.nan],
    "OOB balanced":        [np.nan, np.nan],
    "Optimizado (Grid)":   [cv_results["test_roc_auc"].mean(), cv_results["test_roc_auc"].std()],
    "ExtraTrees":          [et_cv["test_roc_auc"].mean(),      et_cv["test_roc_auc"].std()],
    f"Umbral F1={thr_f1:.2f}": [np.nan, np.nan],
}, index=["CV ROC-AUC (mean)", "CV ROC-AUC (std)"]).T.round(4)

resumen = pd.concat([resumen, cv_summary], axis=1)
print(resumen.to_string())
print(f"\n★ Mejor modelo (test ROC-AUC): {resumen['ROC-AUC'].idxmax()} "
      f"con AUC = {resumen['ROC-AUC'].max():.4f}")


## 16. Predicción para un Niño Nuevo

La predicción se hace **directamente en memoria** con los objetos ya
entrenados en el notebook (`rf_best`, `label_encoders`, `le_y`). No hace
falta serializar a disco: los modelos están vivos en el kernel.

Reutilizamos los encoders persistidos en `label_encoders` — si llega una
categoría no vista en entrenamiento, se lanza un error descriptivo en vez de
generar un *mapping* incorrecto.


In [ ]:
def predecir_anemia(nino_dict,
                     model=rf_best,
                     label_encoders=label_encoders,
                     target_encoder=le_y,
                     threshold=thr_f1):
    """
    Predice la probabilidad de anemia para un nuevo niño.
    Usa el modelo y los encoders YA ajustados en este notebook (no reentrena nada).

    nino_dict debe tener las mismas claves que las columnas de X:
        Edad_meses, Sexo, Peso_kg, Talla_cm,
        ZScore_Talla_Edad, ZScore_Peso_Edad, ZScore_Peso_Talla, ZScore_IMC_Edad,
        Educacion_Madre_Nivel, Educacion_Madre_Anios,
        Intervalo_Nacimiento_Meses, Es_Primerizo, Orden_Nacimiento, Mes_Nacimiento
    """
    fila = pd.DataFrame([nino_dict])[X.columns]

    for col, le in label_encoders.items():
        valor = str(fila.at[0, col])
        if valor not in le.classes_:
            raise ValueError(
                f"Valor '{valor}' no reconocido para la columna '{col}'. "
                f"Valores válidos: {list(le.classes_)}"
            )
        fila[col] = le.transform([valor])

    prob_anemia = model.predict_proba(fila)[:, 1][0]
    clase_idx = int(prob_anemia >= threshold)
    clase = target_encoder.inverse_transform([clase_idx])[0]

    return {
        "prediccion": clase,
        "probabilidad_anemia": round(float(prob_anemia), 4),
        "umbral_usado": round(float(threshold), 4),
    }


# Ejemplo de uso
nino_ejemplo = {
    "Edad_meses": 18,
    "Sexo": "Mujer",
    "Peso_kg": 9.8,
    "Talla_cm": 78.5,
    "ZScore_Talla_Edad": -1.8,
    "ZScore_Peso_Edad": -1.2,
    "ZScore_Peso_Talla": -0.5,
    "ZScore_IMC_Edad": -0.6,
    "Educacion_Madre_Nivel": "Secundaria",
    "Educacion_Madre_Anios": 11,
    "Intervalo_Nacimiento_Meses": 24,
    "Es_Primerizo": 0,
    "Orden_Nacimiento": 2,
    "Mes_Nacimiento": 5,
}

resultado = predecir_anemia(nino_ejemplo)
print("=" * 60)
print("PREDICCIÓN — NIÑO DE EJEMPLO")
print("=" * 60)
for k, v in resultado.items():
    print(f"{k}: {v}")
